# 01 · Obtención de datos: precios internacionales y exportaciones de café y cacao

**Proyecto:** Precios internacionales y desempeño de las exportaciones peruanas de café y cacao, 2015–2025.

**Pregunta de investigación:** ¿Cuánto del crecimiento del valor exportado de café y cacao entre 2015 y 2025 se explica por los
precios internacionales y cuánto por el volumen? ¿Qué tan diversificados están sus mercados de destino (Unión Europea y otros
continentes)?

Este notebook descarga las fuentes oficiales y deja extractos livianos en `datos/originales/`.

| Fuente | Contenido | Uso |
|---|---|---|
| SUNAT, Nota Tributaria, **Cuadro G8** | Exportación definitiva por país de destino y subpartida nacional, mensual, 2005 a agosto 2026 | Valor FOB, volumen y destinos |
| SUNAT, Nota Tributaria, **Cuadro G9** | Exportación definitiva por subpartida nacional, anual | Validación de totales |
| Banco Mundial, **Pink Sheet** (CMO Historical Data, mensual) | Precios internacionales de café arábica, café robusta y cacao, US$/kg, 1960 a la fecha | Precios de referencia |

- SUNAT: <https://www.sunat.gob.pe/estadisticasestudios/exportaciones.html>
- Banco Mundial: <https://www.worldbank.org/en/research/commodity-markets>

Las cifras de SUNAT son **exportaciones definitivas regularizadas** según la fecha de embarque, en **US$ FOB** y **kilogramos**. Los
encabezados del Excel dicen "miles de US$" y "toneladas", pero al comparar con el total nacional se ve que las cifras están en US$ y kg
(ver la sección 4).

In [1]:
import json
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from openpyxl import load_workbook

RAIZ = Path.cwd()
ORIGINALES = RAIZ / "datos" / "originales"
CARPETA_SUNAT = ORIGINALES / "sunat_nota_tributaria"
CARPETA_BM = ORIGINALES / "banco_mundial"
for carpeta in (CARPETA_SUNAT, CARPETA_BM):
    carpeta.mkdir(parents=True, exist_ok=True)

URL_SUNAT = "https://www.sunat.gob.pe/estadisticasestudios/nota_tributaria"
CUADROS = {
    "G8": "cdro_G8.xlsx",  # país de destino × subpartida nacional, mensual (~175 MB)
    "G9": "cdro_G9.xlsx",  # subpartida nacional, anual (~7 MB)
}
PAGINA_BM = "https://www.worldbank.org/en/research/commodity-markets"
# Enlace vigente al momento de la descarga. El Banco Mundial lo cambia en cada actualización, por eso primero se busca en la página.
URL_BM_RESPALDO = (
    "https://thedocs.worldbank.org/en/doc/74e8be41ceb20fa0da750cda2f6b9e4e-0050012026/related/CMO-Historical-Data-Monthly.xlsx"
)
CABECERAS = {"User-Agent": "Mozilla/5.0 (proyecto academico; Python requests)"}

## 1. Descarga

El Cuadro G8 pesa unos 175 MB y la descarga desde SUNAT puede tardar varios minutos. Si el archivo ya existe, no se vuelve a descargar.
Este archivo **no se sube a GitHub** (supera el límite de 100 MB). Se versiona el extracto en CSV que se genera más abajo.

In [2]:
def descargar(url: str, destino: Path, forzar: bool = False) -> Path:
    if destino.exists() and destino.stat().st_size > 0 and not forzar:
        print(f"Ya existe: {destino.name} ({destino.stat().st_size / 1e6:.1f} MB)")
        return destino
    print(f"Descargando {url} ...")
    with requests.get(url, stream=True, timeout=900, headers=CABECERAS) as r:
        r.raise_for_status()
        with open(destino, "wb") as f:
            for bloque in r.iter_content(chunk_size=1 << 20):
                f.write(bloque)
    print(f"Listo: {destino.name} ({destino.stat().st_size / 1e6:.1f} MB)")
    return destino


def url_pink_sheet() -> str:
    """Busca en la página del Banco Mundial el enlace actual del Excel mensual (Pink Sheet)."""
    try:
        pagina = requests.get(PAGINA_BM, timeout=60, headers=CABECERAS).text
        enlaces = re.findall(r'https?://[^"\']+CMO-Historical-Data-Monthly\.xlsx', pagina)
        if enlaces:
            return enlaces[0]
    except requests.RequestException as error:
        print("No se pudo leer la página del Banco Mundial:", error)
    return URL_BM_RESPALDO


archivos = {clave: descargar(f"{URL_SUNAT}/{nombre}", CARPETA_SUNAT / nombre) for clave, nombre in CUADROS.items()}
URL_BM = url_pink_sheet()
archivos["PINK"] = descargar(URL_BM, CARPETA_BM / "CMO-Historical-Data-Monthly.xlsx")

urls = {**{clave: f"{URL_SUNAT}/{nombre}" for clave, nombre in CUADROS.items()}, "PINK": URL_BM}
metadatos = {
    clave: {
        "url": urls[clave],
        "archivo": str(ruta.relative_to(RAIZ)).replace("\\", "/"),
        "fecha_descarga": datetime.fromtimestamp(ruta.stat().st_mtime).strftime("%Y-%m-%d"),
        "tamano_mb": round(ruta.stat().st_size / 1e6, 1),
    }
    for clave, ruta in archivos.items()
}
metadatos

Ya existe: cdro_G8.xlsx (175.7 MB)
Ya existe: cdro_G9.xlsx (6.9 MB)


Ya existe: CMO-Historical-Data-Monthly.xlsx (0.6 MB)


{'G8': {'url': 'https://www.sunat.gob.pe/estadisticasestudios/nota_tributaria/cdro_G8.xlsx',
  'archivo': 'datos/originales/sunat_nota_tributaria/cdro_G8.xlsx',
  'fecha_descarga': '2026-10-31',
  'tamano_mb': 175.7},
 'G9': {'url': 'https://www.sunat.gob.pe/estadisticasestudios/nota_tributaria/cdro_G9.xlsx',
  'archivo': 'datos/originales/sunat_nota_tributaria/cdro_G9.xlsx',
  'fecha_descarga': '2026-10-31',
  'tamano_mb': 6.9},
 'PINK': {'url': 'https://thedocs.worldbank.org/en/doc/74e8be41ceb20fa0da750cda2f6b9e4e-0050012026/related/CMO-Historical-Data-Monthly.xlsx',
  'archivo': 'datos/originales/banco_mundial/CMO-Historical-Data-Monthly.xlsx',
  'fecha_descarga': '2026-11-01',
  'tamano_mb': 0.6}}

## 2. Productos: qué subpartidas se extraen

| Producto | Partidas del Sistema Armonizado |
|---|---|
| **Café** | 0901: café verde (sin tostar), descafeinado, tostado, cáscara y cascarilla |
| **Cacao** | 1801 grano · 1802 cáscara · 1803 pasta · 1804 manteca · 1805 polvo · 1806 chocolate y preparaciones |

Para comparar con los precios internacionales se usa el **producto base** de cada cadena: café verde (0901.11) y cacao en grano
(1801). Son los productos que cotizan en las bolsas de Nueva York y Londres.

In [3]:
PREFIJOS = ("0901", "1801", "1802", "1803", "1804", "1805", "1806")
MESES = {
    "enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6, "julio": 7,
    "agosto": 8, "setiembre": 9, "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12,
}


def es_numero(valor) -> bool:
    return isinstance(valor, (int, float)) and not isinstance(valor, bool)


def codigo_10(valor) -> str:
    """El Excel guarda la subpartida como número y pierde el cero inicial (901119000 -> 0901119000)."""
    return str(int(valor)).zfill(10)


def familia(codigo: str) -> str:
    return "Café" if codigo.startswith("0901") else "Cacao"

## 3. Lectura del Cuadro G8 (país × subpartida × mes)

Cada hoja es un año. Hay filas de **país** (columna C) seguidas de las filas de **subpartidas** (columna E: código, F: descripción).
Para cada mes hay tres columnas: valor FOB, peso neto y peso bruto. La hoja 2026 solo trae los meses ya publicados (enero–agosto al
momento de la descarga). Por eso los meses se leen del encabezado y no se asumen fijos. La lectura tarda unos minutos.

In [4]:
def columnas_por_mes(encabezado) -> dict[int, int]:
    """Devuelve {mes: índice de la columna 'Valor FOB'} leyendo la fila de nombres de meses."""
    return {MESES[str(v).strip().lower()]: i for i, v in enumerate(encabezado) if v and str(v).strip().lower() in MESES}


def leer_g8(ruta: Path) -> pd.DataFrame:
    libro = load_workbook(ruta, read_only=True)
    filas = []
    for hoja in libro.worksheets:
        anio = int(hoja.title)
        cols, pais = None, None
        for fila in hoja.iter_rows(values_only=True):
            if cols is None:
                if any(str(v).strip().lower() == "enero" for v in fila if v):
                    cols = columnas_por_mes(fila)
                continue
            etiqueta, codigo, descripcion = fila[2], fila[4], fila[5]
            valores = {mes: fila[i:i + 3] for mes, i in cols.items()}
            if isinstance(etiqueta, str) and etiqueta.strip() and any(es_numero(v[0]) for v in valores.values()):
                pais = None if etiqueta.strip() == "Total general" else etiqueta.strip()
            elif es_numero(codigo) and pais:
                cod = codigo_10(codigo)
                if not cod.startswith(PREFIJOS):
                    continue
                for mes, (fob, neto, bruto) in valores.items():
                    if fob or neto:
                        filas.append((anio, mes, pais, cod, str(descripcion).strip(), fob or 0, neto or 0, bruto or 0))
        if anio % 5 == 0 or anio == int(libro.sheetnames[-1]):
            print(f"  {anio}: {len(filas):>7,} filas acumuladas · meses {min(cols)}–{max(cols)}")
    libro.close()
    columnas = ["anio", "mes", "pais_sunat", "subpartida", "descripcion", "fob_usd", "peso_neto_kg", "peso_bruto_kg"]
    return pd.DataFrame(filas, columns=columnas)


g8 = leer_g8(archivos["G8"])
g8.shape

  2005:     783 filas acumuladas · meses 1–12


  2010:   5,871 filas acumuladas · meses 1–12


  2015:  12,760 filas acumuladas · meses 1–12


  2020:  21,924 filas acumuladas · meses 1–12


  2025:  32,736 filas acumuladas · meses 1–12


  2026:  34,199 filas acumuladas · meses 1–8


(34199, 8)

In [5]:
g8.sample(6, random_state=1)

,anio,mes,pais_sunat,subpartida,descripcion,fob_usd,peso_neto_kg,peso_bruto_kg
26471,2023,4,"COREA (SUR), REPUBLICA DE",0901119000,"DEMAS CAFES SIN DESCAFEINAR, SIN TOSTAR",22034.450,915.000,956.000
4771,2010,8,ESTADOS UNIDOS,1806320000,CHOCOLATES Y SUS PREPARAC. EN BLOQUES. TABLETA...,434.500,51.489,54.131
16732,2018,7,CHILE,1803100000,PASTA DE CACAO SIN DESGRASAR,76000.000,20000.000,20420.000
11458,2015,4,CANADA,1801001900,"DEMAS CACAO CRUDO EN GRANO, ENTERO O PARTIDO",44100.020,6450.010,6705.480
33141,2026,3,AUSTRALIA,1801001900,"DEMAS CACAO CRUDO EN GRANO, ENTERO O PARTIDO",50683.850,12530.000,12670.000
29656,2024,11,BELGICA,0901119000,"DEMAS CAFES SIN DESCAFEINAR, SIN TOSTAR",9029269.436,1703703.700,1718870.000


## 4. Validación con el Cuadro G9 (subpartida, anual)

El G9 trae el total anual de cada subpartida sin desagregar por país. Si el G8 está completo, la suma del G8 por producto y año debe
coincidir con el G9.

In [6]:
def leer_g9(ruta: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    libro = load_workbook(ruta, read_only=True)
    registros, totales = [], []
    for hoja in libro.worksheets:
        anio = int(hoja.title)
        for fila in hoja.iter_rows(values_only=True):
            if len(fila) < 10:
                continue
            if fila[2] == "Total general":
                totales.append((anio, fila[7], fila[8]))
            elif es_numero(fila[2]) and es_numero(fila[3]):
                cod = codigo_10(fila[3])
                if cod.startswith(PREFIJOS):
                    registros.append((anio, cod, str(fila[4]).strip(), fila[7] or 0, fila[8] or 0))
    libro.close()
    g9 = pd.DataFrame(registros, columns=["anio", "subpartida", "descripcion", "fob_usd", "peso_neto_kg"])
    total = pd.DataFrame(totales, columns=["anio", "fob_usd_total_peru", "peso_neto_kg_total_peru"])
    return g9, total


g9, g9_total = leer_g9(archivos["G9"])
g9_total.tail(3)

,anio,fob_usd_total_peru,peso_neto_kg_total_peru
18,2023,6.428865e+10,5.788482e+10
19,2024,7.437204e+10,6.279829e+10
20,2025,9.029908e+10,6.033535e+10


**Unidades.** El total nacional de 2024 en el G9 sale cerca de 75 000 millones. Esa es la cifra en **dólares** de las exportaciones
peruanas de ese año, no en miles de dólares. Por eso se tratan los valores como US$ y kg.

In [7]:
comparacion = (
    g8.assign(producto=g8.subpartida.map(familia)).groupby(["anio", "producto"]).fob_usd.sum().rename("fob_g8").to_frame()
    .join(g9.assign(producto=g9.subpartida.map(familia)).groupby(["anio", "producto"]).fob_usd.sum().rename("fob_g9"))
)
comparacion["diferencia_%"] = (comparacion.fob_g8 / comparacion.fob_g9 - 1) * 100
assert comparacion["diferencia_%"].dropna().abs().max() < 0.1, "El G8 no cuadra con el G9"
(comparacion.assign(fob_g8=comparacion.fob_g8 / 1e6, fob_g9=comparacion.fob_g9 / 1e6)
 .unstack("producto").loc[2015:].round(1))

fob_g8          fob_g9         diferencia_%     
producto   Cacao    Café   Cacao    Café        Cacao Café
anio                                                      
2015       266.6   613.7   266.6   613.7          0.0  0.0
2016       294.2   762.6   294.2   762.6          0.0 -0.0
2017       236.8   713.9   236.8   713.9          0.0 -0.0
2018       267.5   682.8   267.5   682.8          0.0  0.0
2019       295.3   637.3   295.3   637.3          0.0  0.0
2020       279.5   642.2   279.5   642.2          0.0  0.0
2021       304.0   769.7   304.0   769.7          0.0  0.0
2022       334.0  1236.1   334.0  1236.1          0.0 -0.0
2023       427.7   819.7   427.7   819.7          0.0 -0.0
2024      1280.5  1101.7  1280.5  1101.7          0.0  0.0
2025      1618.3  1802.0  1618.5  1802.0         -0.0 -0.0
2026       579.2   697.1     NaN     NaN          NaN  NaN

Entre 2015 y 2025, la suma del G8 coincide con el G9 en café y cacao (diferencia menor a 0,1 %). El G9 todavía no publica 2026.

## 5. Precios internacionales (Banco Mundial, Pink Sheet)

La hoja `Monthly Prices` trae un precio por mes desde 1960, en dólares corrientes:

- **Café arábica:** precio indicativo de la Organización Internacional del Café (OIC) para los "otros suaves" (*other mild Arabicas*),
  promedio de Nueva York y Bremen/Hamburgo. Es el grupo al que pertenece el café peruano.
- **Café robusta:** precio indicativo OIC para robustas. Solo como referencia: el Perú exporta casi solo arábica.
- **Cacao:** precio diario de la Organización Internacional del Cacao (ICCO), promedio de los futuros de Nueva York y Londres.

Todos están en **US$/kg**, la misma unidad que el valor unitario de exportación (FOB / peso neto).

In [8]:
SERIES_BM = {"Coffee, Arabica": "cafe_arabica", "Coffee, Robusta": "cafe_robusta", "Cocoa": "cacao"}


def leer_pink_sheet(ruta: Path) -> tuple[pd.DataFrame, str]:
    libro = load_workbook(ruta, read_only=True)
    hoja = libro["Monthly Prices"]
    filas = list(hoja.iter_rows(values_only=True))
    libro.close()
    actualizado = next(str(f[0]) for f in filas[:6] if f[0] and str(f[0]).startswith("Updated"))
    i_nombres = next(i for i, f in enumerate(filas) if "Cocoa" in f)
    indices = {SERIES_BM[n]: j for j, n in enumerate(filas[i_nombres]) if n in SERIES_BM}
    unidades = {nombre: filas[i_nombres + 1][j] for nombre, j in indices.items()}
    print("Unidades:", unidades)
    registros = []
    for f in filas[i_nombres + 2:]:
        if not (isinstance(f[0], str) and re.fullmatch(r"\d{4}M\d{2}", f[0])):
            continue
        anio, mes = int(f[0][:4]), int(f[0][5:])
        registros.append({"anio": anio, "mes": mes, **{n: (f[j] if es_numero(f[j]) else None) for n, j in indices.items()}})
    precios = pd.DataFrame(registros)
    fecha = datetime.strptime(actualizado.replace("Updated on", "").strip(), "%B %d, %Y").strftime("%Y-%m-%d")
    return precios, fecha


precios, actualizado_bm = leer_pink_sheet(archivos["PINK"])
metadatos["PINK"]["actualizado_por_bm"] = actualizado_bm
print("Actualizado por el Banco Mundial:", actualizado_bm, "· último mes:", precios.iloc[-1][["anio", "mes"]].astype(int).tolist())
precios.query("anio >= 2015").groupby("anio")[["cafe_arabica", "cafe_robusta", "cacao"]].mean().round(2)

Unidades: {'cacao': '($/kg)', 'cafe_arabica': '($/kg)', 'cafe_robusta': '($/kg)'}
Actualizado por el Banco Mundial: 2026-10-02 · último mes: [2026, 9]


,cafe_arabica,cafe_robusta,cacao
anio,,,
2015,3.53,1.94,3.14
2016,3.61,1.95,2.89
2017,3.32,2.22,2.03
2018,2.92,1.87,2.29
2019,2.88,1.62,2.34
2020,3.32,1.52,2.37
2021,4.51,1.98,2.43
2022,5.63,2.29,2.39
2023,4.54,2.63,3.28


El café arábica pasó de unos 3,5 US$/kg en 2015 a cerca de 8,5 en 2025, y el cacao saltó en 2024. Se guarda la serie completa desde
1960 para poder ubicar esos niveles en la historia. En el notebook 03 se comparan con el valor unitario de las exportaciones peruanas.

## 6. Guardar extractos

In [9]:
g8.to_csv(ORIGINALES / "extracto_g8_cafe_cacao_mensual.csv", index=False)
g9.to_csv(ORIGINALES / "extracto_g9_cafe_cacao_anual.csv", index=False)
g9_total.to_csv(ORIGINALES / "extracto_g9_total_peru_anual.csv", index=False)
precios.to_csv(ORIGINALES / "extracto_precios_internacionales_mensual.csv", index=False)
(ORIGINALES / "metadatos_descarga.json").write_text(json.dumps(metadatos, indent=2, ensure_ascii=False), encoding="utf-8")
for archivo in sorted(ORIGINALES.glob("*.*")):
    print(f"{archivo.name:<48} {archivo.stat().st_size / 1e6:6.2f} MB")

extracto_g8_cafe_cacao_mensual.csv                 3.71 MB
extracto_g9_cafe_cacao_anual.csv                   0.05 MB
extracto_g9_total_peru_anual.csv                   0.00 MB
extracto_precios_internacionales_mensual.csv       0.02 MB
metadatos_descarga.json                            0.00 MB
